# CKP02 — DocMind RAG · Treino de academia

**Prompt Engineering & AI · FIAP · 2º Semestre 2026 · Módulo 2**

**Integrantes:**
Luan de Araujo Carneiro (RM573691) · Pedro Sampaio Mochnacs Arruda (RM573522) ·
Raul Sampaio Mochnacs Arruda (RM573523) · Pedro Ribeiro Lopes (RM570083) ·
Kevin Rodrigues de Melo (RM571777) · Pedro Vianna (RM570747) · Lana Ozeki (RM569795)

Pipeline RAG sobre documentos reais de treino e atividade física, integrado ao assistente
**Halter** do CKP01:

`load → split → embed → store → retrieve → rerank → generate`

| Etapa | Implementação |
|---|---|
| Load | `app/rag/carregador.py` — PDF/TXT/MD com metadados |
| Split | `app/rag/divisor.py` — `RecursiveCharacterTextSplitter`, chunk 256 / 512 / 1024, overlap 12,5% |
| Embed | `app/rag/vetores.py` — `OllamaEmbeddings(model='nomic-embed-text')` |
| Store | `app/rag/vetores.py` — `chromadb.PersistentClient`, coleção `treino_academia_<chunk>` |
| Retrieve | busca vetorial + metadata filtering (`where=`) |
| Rerank | `app/rag/reranker.py` — cross-encoder `ms-marco-MiniLM-L-6-v2` |
| Generate | `app/rag/pipeline.py` — `gemma4:cloud`, temperatura 0, resposta citando o chunk |
| Avaliação | `app/rag/avaliacao.py` — RAGAS faithfulness + answer_relevancy |

## 1. Ambiente

No Colab, cadastre a chave da Ollama Cloud em **Secrets** (ícone de chave na barra lateral)
com o nome `OLLAMA_API_KEY` e libere o acesso para este notebook. A célula abaixo clona o
projeto, instala as dependências e carrega a chave. Rodando localmente, ela usa o `.env`.

In [ ]:
import os
import sys

REPO = "https://github.com/pedrosampaiom2007-gif/cp2.git"

if "google.colab" in sys.modules:
    if not os.path.exists("cp2"):
        !git clone -q {REPO}
    %cd cp2
    !pip install -q -r requirements.txt
    from google.colab import userdata
    os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

sys.path.insert(0, os.getcwd())

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

from app.config import carregar_config

config = carregar_config()
print("Chat:", config.modelo, "| temperatura RAG:", config.rag_temperatura)
print("Embedding:", config.modelo_embedding)

## 2. Base de conhecimento — fontes reais

Cinco documentos públicos do domínio, dois guias oficiais e três artigos científicos
revisados por pares. De cada PDF entram no máximo 5 páginas — as que mais concentram os
termos do tema (`app/rag/fontes.py`).

| # | Documento | Autoria | Link |
|---:|---|---|---|
| 1 | Guia de Atividade Física para a População Brasileira (2021) | Ministério da Saúde | <https://bvsms.saude.gov.br/bvs/publicacoes/guia_atividade_fisica_populacao_brasileira.pdf> |
| 2 | Diretrizes da OMS para atividade física e comportamento sedentário: num piscar de olhos (2020) | Organização Mundial da Saúde | <https://iris.who.int/bitstream/handle/10665/337001/9789240014886-por.pdf> |
| 3 | Recuperação entre séries no treino de força: revisão sistemática e meta-análise | Rev. Bras. Med. Esporte | <https://www.scielo.br/j/rbme/a/Y9vYkwhHhbzKcKNSG9Ft85s/?format=pdf&lang=pt> |
| 4 | Influência de variáveis do treinamento contra-resistência sobre a força muscular de idosos | Rev. Bras. Med. Esporte | <https://www.scielo.br/j/rbme/a/8z4PZxrP4fPvJgfccndzx8M/?format=pdf&lang=pt> |
| 5 | Sessão de treinamento de força supervisionada aumenta a carga total levantada e as respostas subjetivas em sujeitos treinados | Journal of Physical Education | <https://www.scielo.br/j/jpe/a/5fnPtNjMh8Swt3g8kcHTgkt/?format=pdf&lang=pt> |

In [ ]:
from app.rag.baixar_documentos import baixar_todas

registro = baixar_todas()
pd.DataFrame(registro.values())[["id", "tipo", "publico", "ano", "paginas_originais", "total_paginas_original"]]

## 3. Load

In [ ]:
from app.rag.carregador import carregar_documentos, resumo_documentos

documentos = carregar_documentos()
print(f"{len(documentos)} páginas carregadas")
display(pd.DataFrame(resumo_documentos(documentos)))
documentos[0].metadata

## 4. Split — três configurações de chunking

`RecursiveCharacterTextSplitter` com `separators=["\n\n", "\n", ". ", " ", ""]` e overlap de
12,5% do `chunk_size`.

In [ ]:
from app.rag.divisor import CONFIGURACOES, dividir_documentos, estatisticas_chunks

linhas = []
chunks_por_config = {}
for tamanho, cfg in CONFIGURACOES.items():
    chunks = dividir_documentos(documentos, tamanho)
    chunks_por_config[tamanho] = chunks
    linhas.append({"chunk_size": tamanho, "chunk_overlap": cfg.chunk_overlap, **estatisticas_chunks(chunks)})
pd.DataFrame(linhas)

In [ ]:
exemplo = chunks_por_config[512][3]
print(exemplo.metadata["chunk_id"], "|", exemplo.metadata["titulo"], "| p.", exemplo.metadata["pagina"])
print("-" * 80)
print(exemplo.page_content)

## 5. Embed — nomic-embed-text

In [ ]:
import numpy as np

from app.rag.vetores import PREFIXO_CONSULTA, PREFIXO_DOCUMENTO, criar_embeddings

embeddings = criar_embeddings(config)

frases = [
    "Adultos devem fazer atividades de fortalecimento muscular dois ou mais dias por semana.",
    "Treinar musculação pelo menos duas vezes na semana é recomendado para adultos.",
    "O intervalo entre séries influencia o volume total do treino.",
]
vetores_docs = np.array(embeddings.embed_documents([PREFIXO_DOCUMENTO + f for f in frases]))
vetor_consulta = np.array(embeddings.embed_query(PREFIXO_CONSULTA + "quantas vezes por semana fazer musculação?"))

def cosseno(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

print("Dimensão do embedding:", len(vetor_consulta))
pd.DataFrame({"frase": frases, "similaridade_com_a_consulta": [round(cosseno(vetor_consulta, v), 3) for v in vetores_docs]})

## 6. Store — ChromaDB local

In [ ]:
from app.rag.pipeline import DocMindRAG
from app.rag.reranker import Reranker

reranker = Reranker()
bases = {}
for tamanho in CONFIGURACOES:
    rag = DocMindRAG(chunk_size=tamanho, config=config, reranker=reranker, documentos=documentos)
    rag.preparar()
    bases[tamanho] = rag
    print(f"coleção {rag.base.nome_colecao}: {rag.base.total()} chunks")

## 7. Retrieve — busca semântica

In [ ]:
rag = bases[config.rag_chunk_size]

def mostrar(trechos):
    return pd.DataFrame(
        [{"fonte": t.titulo[:60], "pagina": t.pagina, "chunk": t.chunk_id,
          "similaridade": t.similaridade, "rerank": t.score_rerank, "trecho": t.texto[:160]}
         for t in trechos]
    )

consulta = "Quantos minutos por semana de atividade física moderada um adulto deve fazer?"
mostrar(rag.recuperar(consulta, reranking=False))

## 8. Metadata filtering

Cada chunk carrega `tipo`, `categoria`, `publico`, `ano` e `id` da fonte. O filtro vira o
`where=` da consulta no Chroma.

In [ ]:
from app.rag.vetores import montar_filtro

filtro = montar_filtro(tipo="artigo_cientifico")
print(filtro)
mostrar(rag.recuperar("intervalo de descanso entre séries", filtros=filtro, reranking=False))

In [ ]:
filtro = montar_filtro(publico="idosos", ano_minimo=2000)
print(filtro)
mostrar(rag.recuperar("quantas séries e qual intensidade para ganhar força?", filtros=filtro, reranking=False))

## 9. Reranking com cross-encoder

O Chroma traz 12 candidatos e o cross-encoder escolhe os 4 que vão para o LLM.

In [ ]:
consulta = "Qual intervalo entre séries favorece a hipertrofia?"
print("Antes (ordem da busca vetorial):")
display(mostrar(rag.recuperar(consulta, reranking=False)))
print("Depois do reranking:")
display(mostrar(rag.recuperar(consulta, reranking=True)))

## 10. Generate — resposta citando o chunk de origem

In [ ]:
from app.rag.avaliacao import PERGUNTAS_TESTE

for pergunta in PERGUNTAS_TESTE[:5]:
    resposta = rag.responder(pergunta)
    display(Markdown(f"### {pergunta}\n\n{resposta.para_markdown()}"))

### `buscar(consulta)` — função usada como tool no CKP03

Devolve os trechos mais relevantes já com a fonte, prontos para o agente usar.

In [ ]:
from app.rag.pipeline import buscar

print(buscar("O que é recomendado sobre o tempo sentado?"))

## 11. RAGAS — comparação das configurações de chunking

As mesmas 8 perguntas passam pelas três configurações. O juiz do RAGAS é o próprio
`gemma4:cloud` (temperatura 0) e os embeddings são do `nomic-embed-text`.

In [ ]:
from app.rag.avaliacao import (
    avaliar_configuracao,
    justificativa,
    salvar_resultados,
    tabela_por_pergunta,
    tabela_resumo,
)

resultados = [
    avaliar_configuracao(tamanho, PERGUNTAS_TESTE, config, reranker=reranker, documentos=documentos)
    for tamanho in CONFIGURACOES
]
salvar_resultados(resultados)
tabela_resumo(resultados)

In [ ]:
tabela_por_pergunta(resultados)

In [ ]:
import matplotlib.pyplot as plt

resumo = tabela_resumo(resultados).set_index("chunk_size")
ax = resumo[["faithfulness_medio", "answer_relevancy_medio"]].plot.bar(figsize=(7, 4), rot=0)
ax.axhline(0.7, linestyle="--", linewidth=1, color="gray")
ax.set_ylim(0, 1)
ax.set_ylabel("média RAGAS")
ax.set_title("Chunking × RAGAS")
plt.show()

In [ ]:
display(Markdown("### Escolha final\n\n" + justificativa(resultados)))

In [ ]:
pd.concat([r.tabela for r in resultados])[["chunk_size", "pergunta", "resposta", "fontes", "faithfulness"]]

## 12. Interface Gradio

A aba **📚 DocMind RAG** da interface do Halter traz o chat com o RAG, os filtros de
metadados, o liga/desliga do reranking e as fontes citadas em cada resposta.

In [ ]:
from app.main import Aplicacao, construir_interface

aplicacao = Aplicacao()
aplicacao.bases_rag.update(bases)
demo = construir_interface(aplicacao)
demo.launch(share="google.colab" in sys.modules, inline=False, prevent_thread_lock=True)